# plus2net.com Webpage Quality & Editorial Leak Auditor
This modular notebook audits webpages for accidental AI editorial leaks, structural jargon (`hub`, `spoke`, `pillar page`), and tracking identifiers (`site 6`, `previous version`) using the **Gemini API**.

### Setup Instructions:
1. Click the **Key icon (Secrets)** in the left sidebar of Google Colab.
2. Add a new secret named `GEMINI_API_KEY` with your API key.
3. Toggle **Notebook access** ON for `GEMINI_API_KEY`.

In [ ]:
# Cell 1: Dependencies Installation
!pip install -q google-genai beautifulsoup4 requests
print("[✓] Required packages installed successfully.")

In [ ]:
# Cell 2: Authentication & Client Setup
from google import genai
from google.colab import userdata

try:
    api_key = userdata.get('GEMINI_API_KEY')
    client = genai.Client(api_key=api_key)
    print("[✓] Gemini client initialized successfully using Colab Secret.")
except Exception as e:
    print(f"[x] Error retrieving API key from Colab Secrets: {e}")
    print("Please make sure you saved 'GEMINI_API_KEY' in the Colab Secrets panel.")

In [ ]:
# Cell 3: Configuration Settings
# Choose mode: 'single' for testing one URL, or 'sitemap' for bulk batch checking
FETCH_MODE = "sitemap"  # Options: "single" or "sitemap"

# Single URL sample for quick testing
SAMPLE_URL = "https://www.plus2net.com/python/tkinter-buttons.php"

# Sitemap source
SITEMAP_URL = "https://www.plus2net.com/sitemap.xml"

# Batch limit (Adjust between 1 to 10 as needed)
BATCH_LIMIT = 5

print(f"[*] Configuration Loaded:")
print(f"    - Mode: {FETCH_MODE}")
print(f"    - Batch Limit: {BATCH_LIMIT} URLs")

In [ ]:
# Cell 4: URL Collector Module
import requests
import xml.etree.ElementTree as ET

def get_target_urls(mode, sample_url, sitemap_url, limit):
    urls = []
    if mode == "single":
        return [sample_url]

    try:
        print(f"[*] Fetching sitemap from {sitemap_url}...")
        response = requests.get(sitemap_url, timeout=10)
        response.raise_for_status()

        root = ET.fromstring(response.content)
        namespace = {'ns': 'http://www.sitemaps.org/schemas/sitemap/0.9'}

        for loc in root.findall('.//ns:loc', namespace):
            if loc.text:
                urls.append(loc.text.strip())
            if len(urls) >= limit:
                break

        if not urls:
            for loc in root.findall('.//{http://www.sitemaps.org/schemas/sitemap/0.9}loc'):
                if loc.text:
                    urls.append(loc.text.strip())
                if len(urls) >= limit:
                    break

        print(f"[✓] Successfully retrieved {len(urls)} URLs from sitemap.")
    except Exception as e:
        print(f"[x] Error fetching sitemap: {e}")

    return urls[:limit]

target_urls = get_target_urls(FETCH_MODE, SAMPLE_URL, SITEMAP_URL, BATCH_LIMIT)
for idx, u in enumerate(target_urls, 1):
    print(f"  {idx}. {u}")

In [ ]:
# Cell 5: Audit Engine Module
import json
from google.genai import types
from bs4 import BeautifulSoup

def clean_html_to_text(html_content):
    """Strips HTML tags and normalizes whitespace."""
    soup = BeautifulSoup(html_content, 'html.parser')
    for script in soup(["script", "style"]):
        script.extract()
    text = soup.get_text(separator=' ')
    return ' '.join(text.split())

def audit_webpage_content(url, client_instance):
    """Fetches a live URL and runs Gemini semantic leak analysis."""
    try:
        headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64)'}
        web_response = requests.get(url, headers=headers, timeout=10)
        if web_response.status_code != 200:
            return {"error": f"HTTP status {web_response.status_code}"}

        plain_text = clean_html_to_text(web_response.text)

        prompt = f"""
        You are a strict technical editor auditing a programming tutorial webpage for plus2net.com.
        Analyze the text snippet below for any "editorial leaks", AI conversational artifacts, or internal structural jargon.
        Specifically look for:
        1. AI internal planning notes or conversational framing (e.g., "Here is the revised draft").
        2. Architecture or site structuring jargon that shouldn't face users (e.g., references to "hub", "spoke", "pillar page" as layout commands — ignore legitimate technical references like GitHub).
        3. Internal processing, citation, or tracking identifiers (e.g., "site 6", "site 7", "[cite: 12]", version notes like "previous version").

        Respond in valid JSON format with exactly these two keys:
        - "has_leak": boolean (true if an editorial leak or anomaly is found, false otherwise)
        - "explanation": string (short reason if true, or "clean" if false)

        Text to analyze:
        {plain_text[:4000]}
        """

        response = client_instance.models.generate_content(
            model='gemini-2.5-flash',
            contents=prompt,
            config=types.GenerateContentConfig(
                response_mime_type="application/json",
                temperature=0.1
            ),
        )

        return json.loads(response.text)

    except Exception as e:
        return {"error": str(e)}

print("[✓] Audit engine functions defined.")

In [ ]:
# Cell 6: Execution Loop
print(f"[*] Starting audit batch on {len(target_urls)} URL(s)...\n" + "="*60)

for idx, url in enumerate(target_urls, 1):
    print(f"[{idx}/{len(target_urls)}] Auditing: {url}")
    result = audit_webpage_content(url, client)

    if "error" in result:
        print(f"  [x] Error: {result['error']}")
    elif result.get('has_leak'):
        print(f"  [!] Flagged: {result.get('explanation')}")
    else:
        print(f"  [✓] Status: Clean")
    print("-" * 60)

print("\n[*] Batch audit complete!")